# Proyecto 2 – Análisis Exploratorio
### Data Science

**Reto 18: Hackeando el cuerpo humano** — [HuBMAP - Hacking the Human Body](https://www.kaggle.com/competitions/hubmap-organ-segmentation)

**División del trabajo (secciones consecutivas, no intercaladas):**
- Persona 1 → secciones 1, 2, 3 y 4.a
- Persona 2 → secciones 4.b y 4.c
- Persona 3 → secciones 4.d y 5


---
## Configuración inicial

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = "data/"

---
## 1. Investigación del tema

HuBMAP es un programa que busca mapear el cuerpo humano a nivel de célula, algo así como armar un mapa detallado de cómo están organizados los tejidos sanos en cada órgano. La competencia de Kaggle nace de ahí y junta datos de HuBMAP con los del Atlas de Proteínas Humanas (HPA), y lo que pide es encontrar y delimitar, dentro de fotos de tejido, las llamadas unidades funcionales de tejido (FTU).

Una FTU es básicamente un grupo pequeño de células organizadas alrededor de un vaso sanguíneo, que en conjunto cumplen una función específica dentro del órgano al que pertenecen. El reto incluye cinco órganos: riñón, próstata, bazo, pulmón e intestino grueso, y en cada uno la FTU se ve distinta porque cada órgano tiene su propia estructura.

Las imágenes vienen de cortes de tejido que se tiñen con una tinción llamada PAS, que resalta ciertas estructuras del tejido para que se puedan distinguir mejor al verlas al microscopio. Como los datos vienen de dos fuentes distintas (HuBMAP y HPA), y cada una prepara y escanea las muestras a su manera, un mismo órgano puede verse un poco diferente dependiendo de dónde salió la imagen.

Todo esto importa porque la idea final es lograr que un modelo reconozca estas unidades de tejido sin importar el órgano ni la fuente de la imagen, y no solo memorice los casos que vio en el entrenamiento. Eso es justo lo que hace difícil el problema, y es la base para poder construir ese mapa completo del cuerpo humano que se mencionaba al principio.

---
## 2. Análisis del problema planteado y los datos

### Situación problemática

Hoy en día identificar las unidades funcionales de tejido en una imagen de biopsia lo hacen patólogos a mano, revisando la muestra al microscopio y marcando dónde están esas estructuras. Esto toma tiempo, depende de la experiencia de la persona que lo hace y no es algo que se pueda escalar a miles de imágenes de distintos órganos. Además, como las muestras pueden venir de laboratorios distintos con procesos de preparación distintos, dos imágenes del mismo órgano no siempre se ven igual, lo que complica todavía más hacer este trabajo de forma manual y consistente.

### Problema científico

El problema se puede plantear como una pregunta de segmentación de imágenes: a partir de una imagen de tejido, ¿se puede entrenar un modelo que marque con precisión en qué zonas de la imagen están las unidades funcionales de tejido, sin importar de qué órgano se trate ni de qué fuente venga la muestra? La dificultad extra es que el dataset solo trae unos cuantos órganos y fuentes de datos, entonces el modelo tiene que aprender un patrón que generalice y no simplemente memorizar cómo se ve cada órgano en particular.

### Objetivos

**Objetivo general:**

Explorar y describir los datos de la competencia HuBMAP para entender cómo están compuestas las imágenes de tejido y las variables que las acompañan, de forma que sirva de base para pensar en una futura solución al problema de segmentación de FTU.

**Objetivos específicos:**
1. Describir las variables numéricas y categóricas del dataset (órgano, fuente de datos, edad, sexo, grosor del tejido, dimensiones de imagen) y detectar diferencias entre grupos.
2. Analizar cómo se relacionan variables como el órgano, la fuente de datos, la edad y el sexo, para identificar qué tan variadas son las imágenes según esos factores.

### Descripción general de los datos

In [4]:
train = pd.read_csv(DATA_DIR + "train.csv")

print("train:", train.shape)
train.head()

train: (351, 10)


,id,organ,data_source,img_height,img_width,pixel_size,tissue_thickness,rle,age,sex
0,10044,prostate,HPA,3000,3000,0.4,4,1459676 77 1462675 82 1465674 87 1468673 92 14...,37.0,Male
1,10274,prostate,HPA,3000,3000,0.4,4,715707 2 718705 8 721703 11 724701 18 727692 3...,76.0,Male
2,10392,spleen,HPA,3000,3000,0.4,4,1228631 20 1231629 24 1234624 40 1237623 47 12...,82.0,Male
3,10488,lung,HPA,3000,3000,0.4,4,3446519 15 3449517 17 3452514 20 3455510 24 34...,78.0,Male
4,10610,spleen,HPA,3000,3000,0.4,4,478925 68 481909 87 484893 105 487863 154 4908...,21.0,Female


El archivo train.csv tiene 351 filas y 10 columnas, cada fila es una imagen de tejido con su información asociada: el órgano al que pertenece, si la muestra viene de HuBMAP o de HPA, el alto y ancho de la imagen, el tamaño de píxel, el grosor del tejido, la máscara de la FTU codificada en formato rle, y la edad y sexo del donante. Solo se usa este archivo porque es el único que trae datos completos, ya que no se va a participar en la competencia como tal.

---
## 3. Limpieza y preprocesamiento

In [5]:
train.isna().sum()

id                  0
organ               0
data_source         0
img_height          0
img_width           0
pixel_size          0
tissue_thickness    0
rle                 0
age                 0
sex                 0
dtype: int64

In [6]:
print("Duplicados en train:", train.duplicated().sum())

Duplicados en train: 0


In [7]:
for col in ["organ", "data_source", "sex"]:
    train[col] = train[col].astype("category")

train.dtypes

id                     int64
organ               category
data_source         category
img_height             int64
img_width              int64
pixel_size           float64
tissue_thickness       int64
rle                   object
age                  float64
sex                 category
dtype: object

No se encontraron valores nulos ni filas duplicadas en train.csv, así que no hubo que eliminar ni imputar nada. Lo único que se hizo fue convertir organ, data_source y sex a tipo categórico, porque pandas las traía como texto plano y así quedan mejor identificadas para el análisis. La columna rle no se decodifica aquí para todo el dataset porque son máscaras pesadas, esa decodificación se hace más adelante, solo sobre la muestra de imágenes que se use para la parte visual.

---
## 4. Análisis exploratorio de los datos

### 4.a Variables y observaciones disponibles

In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351 entries, 0 to 350
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   id                351 non-null    int64   
 1   organ             351 non-null    category
 2   data_source       351 non-null    category
 3   img_height        351 non-null    int64   
 4   img_width         351 non-null    int64   
 5   pixel_size        351 non-null    float64 
 6   tissue_thickness  351 non-null    int64   
 7   rle               351 non-null    object  
 8   age               351 non-null    float64 
 9   sex               351 non-null    category
dtypes: category(3), float64(2), int64(4), object(1)
memory usage: 20.8+ KB


En total train.csv tiene 351 observaciones (una por cada imagen de tejido) y 10 variables. id es solo el identificador de la imagen, no aporta nada analítico por sí solo. organ, data_source y sex son categóricas, ya convertidas a ese tipo en la sección anterior: organ tiene los cinco órganos del reto, data_source indica si la muestra viene de HuBMAP o de HPA, y sex es el sexo del donante. img_height e img_width son numéricas y dan el alto y ancho de la imagen en píxeles. pixel_size y tissue_thickness también son numéricas y describen el tamaño real que representa cada píxel y el grosor de la muestra de tejido. age es numérica y es la edad del donante. Por último, rle es una variable de texto que guarda la máscara de la FTU codificada, no es un valor que se pueda resumir como las demás.

### 4.b Resumen de variables numéricas y tablas de frecuencia de categóricas

#### Resumen de variables numéricas

_Completar interpretación de estadísticos descriptivos (edad, tamaño de píxel, grosor de tejido, alto/ancho de imagen)._

In [ ]:
train.describe()

#### Tablas de frecuencia de variables categóricas

_Completar interpretación (órgano, fuente de datos, sexo)._

In [ ]:
for col in ["organ", "data_source", "sex"]:
    if col in train.columns:
        print(f"\n--- {col} ---")
        print(train[col].value_counts())
        print(train[col].value_counts(normalize=True).round(3))

### 4.c Cruce de variables importantes

#### Cruce de variables clave

_Completar: ej. órgano vs. edad/sexo/grosor de tejido._

#### Correlaciones entre variables numéricas

In [ ]:
numeric_cols = train.select_dtypes(include="number").columns
train[numeric_cols].corr()

#### Outliers y valores faltantes

_Completar: identificación de outliers (ej. boxplots por órgano), explicación de posibles causas, y decisión tomada ante valores faltantes (si los hay)._

### 4.d Gráficos exploratorios

#### Gráficos univariados

_Completar: histogramas y boxplots para numéricas, gráficos de barra para categóricas. Escribir lo que se observa debajo de cada gráfico._

In [ ]:
# Ejemplo de histograma
# train["age"].hist(bins=20)
# plt.title("Distribución de edad")
# plt.show()

#### EDA visual de imágenes

_Completar sobre la muestra de imágenes descargada (no todo el set): ejemplos de tejido por órgano, dimensiones típicas, máscara superpuesta (decodificando la columna `rle`)._

In [ ]:
def rle_decode(rle_string, shape):
    """Decodifica una máscara en formato run-length encoding a un array binario (h, w)."""
    s = rle_string.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        mask[lo:hi] = 1
    return mask.reshape(shape)

---
## 5. Conclusiones

_Completar: resumen de los hallazgos del análisis exploratorio y conclusiones sobre los siguientes pasos a seguir._